# Gold Layer — Phase 1 Build

Builds all **reference dimensions** and **fact tables** from the silver layer.

| Step | Table | Source |
|------|-------|--------|
| 1 | `dim_date` | Generated (2015–2035) |
| 2 | `dim_department` | silver.departments |
| 3 | `dim_medication` | silver.medications |
| 4 | `dim_diagnosis_code` | silver.diagnoses (ICD-10 codes) |
| 5 | `dim_payer` | silver.insurance_claims |
| 6 | `fact_appointments` | silver.encounters (OUTPATIENT/EMERGENCY) |
| 7 | `fact_medical_records` | silver.encounters + silver.diagnoses |
| 8 | `fact_admissions` | silver.admissions |
| 9 | `fact_surgeries` | silver.procedures |
| 10 | `fact_lab_results` | silver.lab_results + silver.lab_orders |
| 11 | `fact_billing` | silver.invoices + silver.payments + silver.insurance_claims |

> **dim_patient** and **dim_doctor** are built by the identity-resolution
> notebooks (01 & 02) and are NOT re-built here.

## 1. Setup

In [ ]:
# No external library installs required.
# PySpark and Delta Lake are natively provided by Databricks Runtime.
print('Environment ready.')

In [ ]:
import sys
sys.path.insert(0, '/Workspace/Users/sallam5100@gmail.com/.bundle/hospital_edw/dev/files')

In [ ]:
from config.gold_config import GOLD_TABLES, SILVER_TABLES, BRIDGE_TABLES, DATE_SPINE_START, DATE_SPINE_END

print('Gold tables :')
for k, v in GOLD_TABLES.items():
    print(f'  {k:<25} → {v}')
print()
print('Bridge maps:')
for k, v in BRIDGE_TABLES.items():
    print(f'  {k:<25} → {v}')

## 2. Verify Prerequisites

Confirm the identity-resolution bridge maps exist before building facts.

In [ ]:
from src.gold.writer import table_exists

prereqs = {
    'dim_patient':  GOLD_TABLES['dim_patient'],
    'dim_doctor':   GOLD_TABLES['dim_doctor'],
    'patient_map':  BRIDGE_TABLES['patient_map'],
    'doctor_map':   BRIDGE_TABLES['doctor_map'],
}

all_ok = True
for name, tbl in prereqs.items():
    exists = table_exists(spark, tbl)
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {name}: {tbl}')
    if not exists:
        all_ok = False

if not all_ok:
    raise RuntimeError(
        'One or more prerequisite tables are missing. '
        'Run the identity-resolution notebooks first.'
    )
print('\nAll prerequisites satisfied ✅')

## 3. Quick Silver Layer Check

Row counts from all silver source tables used by the gold builder.

In [ ]:
print(f"{'Silver Table':<35} {'Rows':>10}")
print('-' * 47)
for name, tbl in SILVER_TABLES.items():
    try:
        n = spark.table(tbl).count()
        print(f"  {name:<33} {n:>10,}")
    except Exception as e:
        print(f"  {name:<33} {'ERROR':>10}  ({e})")

## 4. Build Reference Dimensions

Steps 1–5: dim_date, dim_department, dim_medication, dim_diagnosis_code, dim_payer.

In [ ]:
from src.gold.dimensions import build_all_dimensions

build_all_dimensions(
    spark,
    tables       = GOLD_TABLES,
    silver_tables= SILVER_TABLES,
    date_start   = DATE_SPINE_START,
    date_end     = DATE_SPINE_END,
)

## 5. Build Fact Tables

Steps 6–11: fact_appointments, fact_medical_records, fact_admissions,
fact_surgeries, fact_lab_results, fact_billing.

In [ ]:
from src.gold.facts import build_all_facts

build_all_facts(
    spark,
    tables        = GOLD_TABLES,
    silver_tables = SILVER_TABLES,
    bridge_tables = BRIDGE_TABLES,
)

## 6. Validate — Row Counts & Spot Checks

In [ ]:
from pyspark.sql import functions as F

# ── Dimension counts ─────────────────────────────────────────────
dim_keys = ['dim_date','dim_patient','dim_doctor',
            'dim_department','dim_medication','dim_diagnosis_code','dim_payer']
print('DIMENSIONS')
print(f"  {'Table':<30} {'Rows':>10}")
print('  ' + '-' * 42)
for k in dim_keys:
    tbl = GOLD_TABLES[k]
    n = spark.table(tbl).count()
    print(f"  {k:<30} {n:>10,}")

# ── Fact counts ──────────────────────────────────────────────────
fact_keys = ['fact_appointments','fact_medical_records','fact_admissions',
             'fact_surgeries','fact_lab_results','fact_billing']
print()
print('FACTS')
print(f"  {'Table':<30} {'Rows':>10}")
print('  ' + '-' * 42)
for k in fact_keys:
    tbl = GOLD_TABLES[k]
    n = spark.table(tbl).count()
    print(f"  {k:<30} {n:>10,}")

In [ ]:
# ── Appointments: status breakdown ────────────────────────────────
display(
    spark.table(GOLD_TABLES['fact_appointments'])
    .groupBy('encounter_type', 'appointment_status')
    .agg(
        F.count('*').alias('count'),
        F.countDistinct('enterprise_patient_id').alias('unique_patients'),
    )
    .orderBy('encounter_type', F.desc('count'))
)

In [ ]:
# ── Medical records: top 10 ICD-10 codes ─────────────────────────
display(
    spark.table(GOLD_TABLES['fact_medical_records'])
    .filter(F.col('primary_icd10_code').isNotNull())
    .groupBy('primary_icd10_code', 'primary_diagnosis')
    .agg(F.count('*').alias('visit_count'))
    .orderBy(F.desc('visit_count'))
    .limit(10)
)

In [ ]:
# ── Admissions: LOS distribution ─────────────────────────────────
display(
    spark.table(GOLD_TABLES['fact_admissions'])
    .filter(F.col('length_of_stay_days').isNotNull())
    .groupBy('admission_type')
    .agg(
        F.count('*').alias('admissions'),
        F.round(F.avg('length_of_stay_days'), 1).alias('avg_los_days'),
        F.max('length_of_stay_days').alias('max_los_days'),
    )
    .orderBy(F.desc('admissions'))
)

In [ ]:
# ── Surgeries: top procedures by volume ──────────────────────────
display(
    spark.table(GOLD_TABLES['fact_surgeries'])
    .groupBy('procedure_name')
    .agg(
        F.count('*').alias('surgery_count'),
        F.round(F.avg('duration_minutes'), 1).alias('avg_duration_min'),
    )
    .orderBy(F.desc('surgery_count'))
    .limit(15)
)

In [ ]:
# ── Lab results: abnormal rate by test ───────────────────────────
display(
    spark.table(GOLD_TABLES['fact_lab_results'])
    .groupBy('test_name')
    .agg(
        F.count('*').alias('total_results'),
        F.sum(F.col('is_abnormal').cast('int')).alias('abnormal_count'),
        F.round(
            F.sum(F.col('is_abnormal').cast('int')) * 100.0 / F.count('*'), 1
        ).alias('abnormal_pct'),
    )
    .orderBy(F.desc('total_results'))
)

In [ ]:
# ── Billing: revenue summary ──────────────────────────────────────
display(
    spark.table(GOLD_TABLES['fact_billing'])
    .agg(
        F.count('*').alias('total_invoices'),
        F.round(F.sum('gross_amount'),             2).alias('total_gross_EGP'),
        F.round(F.sum('total_amount_paid'),        2).alias('total_collected_EGP'),
        F.round(F.sum('outstanding_balance'),      2).alias('total_outstanding_EGP'),
        F.round(F.sum('total_insurance_paid'),     2).alias('total_insurance_paid_EGP'),
        F.round(F.avg('length_of_stay_days') if False else F.lit(None), 2)
    )
)

## 7. Referential Integrity Checks

Verify that every fact row has a valid enterprise ID in the dimension tables.

In [ ]:
def ri_check(fact_tbl, fact_col, dim_tbl, dim_col, label):
    fact = spark.table(fact_tbl)
    dim  = spark.table(dim_tbl).select(dim_col).distinct()
    orphans = fact.join(dim, fact[fact_col] == dim[dim_col], how='left_anti')
    n = orphans.count()
    status = '✅' if n == 0 else f'⚠️  {n:,} orphans'
    print(f'  {status}  {label}')

print('Referential integrity:')
ri_check(GOLD_TABLES['fact_appointments'],
         'enterprise_patient_id', GOLD_TABLES['dim_patient'],
         'enterprise_patient_id', 'fact_appointments → dim_patient')

ri_check(GOLD_TABLES['fact_medical_records'],
         'enterprise_patient_id', GOLD_TABLES['dim_patient'],
         'enterprise_patient_id', 'fact_medical_records → dim_patient')

ri_check(GOLD_TABLES['fact_admissions'],
         'enterprise_patient_id', GOLD_TABLES['dim_patient'],
         'enterprise_patient_id', 'fact_admissions → dim_patient')

ri_check(GOLD_TABLES['fact_surgeries'],
         'enterprise_patient_id', GOLD_TABLES['dim_patient'],
         'enterprise_patient_id', 'fact_surgeries → dim_patient')

ri_check(GOLD_TABLES['fact_lab_results'],
         'enterprise_patient_id', GOLD_TABLES['dim_patient'],
         'enterprise_patient_id', 'fact_lab_results → dim_patient')

ri_check(GOLD_TABLES['fact_billing'],
         'enterprise_patient_id', GOLD_TABLES['dim_patient'],
         'enterprise_patient_id', 'fact_billing → dim_patient')

ri_check(GOLD_TABLES['fact_appointments'],
         'dept_id', GOLD_TABLES['dim_department'],
         'dept_id', 'fact_appointments → dim_department')

ri_check(GOLD_TABLES['fact_billing'],
         'payer_key', GOLD_TABLES['dim_payer'],
         'payer_key', 'fact_billing → dim_payer')

## 8. Gold Layer Summary

Star schema is ready. Connect a BI tool (Power BI, Tableau, Databricks SQL)
directly to `hospital_catalog.gold.*`.

```
                        dim_date
                           │
   dim_patient ────── fact_appointments ────── dim_doctor
   dim_patient ────── fact_medical_records ─── dim_doctor
   dim_patient ────── fact_admissions ─────── dim_department
   dim_patient ────── fact_surgeries ─────── dim_doctor (surgeon)
   dim_patient ────── fact_lab_results ──────── dim_doctor
   dim_patient ────── fact_billing ─────────── dim_payer
                                               dim_medication  (prescriptions phase)
                                               dim_diagnosis_code
```

**Phase 2** (after ingesting nurses, helpers, ward, room, bed, staffshift):
- dim_nurse, dim_helper, dim_ward, dim_room, dim_bed
- fact_staff_shifts
- Backfill nurse_id / helper_id in fact_admissions and fact_surgeries

In [ ]:
# Payer breakdown — insured vs self-pay
# SELF-PAY rows are cash/uninsured patients with no insurance claim.
# They receive a sentinel FK so the RI check passes for all rows.
display(
    spark.table(GOLD_TABLES['fact_billing'])
    .groupBy('payer_name')
    .agg(
        F.count('*').alias('invoice_count'),
        F.round(F.sum('gross_amount'), 2).alias('gross_amount_total'),
        F.round(F.sum('total_amount_paid'), 2).alias('collected_total'),
        F.round(F.sum('outstanding_balance'), 2).alias('outstanding_total'),
        F.round(F.sum('total_insurance_paid'), 2).alias('insurance_paid_total'),
    )
    .orderBy(F.desc('invoice_count'))
)